# 9th ABC Challenge - UWB Fall Track: Bi-Level Hybrid Cascade Pipeline
**MobileNetV3 (fast proposal) → 1D-FPN ActionFormer-style decoder (boundary regression) → Temporal DIoU loss → 1D Soft-NMS**

Continuous, unsegmented UWB radar streams: 3 radars · 50 Hz · 368 range bins.

| Cell | Content |
|---|---|
| 1 | Environment, Drive, paths, constants, hyper-parameters |
| 2 | SVD clutter removal · 12 spatial-band features · robust IQR normalisation |
| 3 | Stage 1 - `MobileNetV3ProposalNet` + `extract_suspicious_regions` |
| 4 | Stage 2 - `TemporalFPNDecoder` (stem + 5 dilated residual blocks, FPN, per-frame heads) |
| 5 | `create_regression_targets` + `TemporalDIoULoss` (Focal + 5× splice penalty + 2·DIoU) |
| 6 | Inference engine: `decode_and_nms` (1D Soft-NMS) + cascade runner |
| 6B | Training of both stages (needed to obtain weights; skipped when checkpoints exist) |
| 6C | Validation: Stage-1 recall, Soft-NMS floor sweep, official evaluator |
| 7 | Test streams `test_00 … test_04` → `submission_colab.csv` |

> Expected Drive layout: `MyDrive/ABC27/abc2027_uwb_train_v1/` and `MyDrive/ABC27/abc2027_uwb_test_v1/` (extracted packages). Select a **GPU runtime** (Runtime → Change runtime type).

## Cell 1 - Environment Setup & Paths

In [ ]:
import os, sys, gc, io, math, time, json, random, warnings, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import scipy
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------------------------------------------------------
# 1. Google Drive
# ----------------------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                       # running locally (smoke tests / VS Code)
    IN_COLAB = False

ROOT_DIR = (Path("/content/drive/MyDrive/ABC27") if IN_COLAB
            else Path(os.environ.get("ABC_ROOT", ".")).resolve())

# ----------------------------------------------------------------------------
# 2. Paths (supports extracted folders AND in-memory reading from .zip archives)
# ----------------------------------------------------------------------------
TRAIN_PKG_DIR     = ROOT_DIR / "abc2027_uwb_train_v1"
TEST_PKG_DIR      = ROOT_DIR / "abc2027_uwb_test_v1"
TRAIN_ZIP_PATH    = ROOT_DIR / "abc2027_uwb_train_v1.zip"
TEST_ZIP_PATH     = ROOT_DIR / "abc2027_uwb_test_v1.zip"
EVENTS_CSV        = TRAIN_PKG_DIR / "streams_events.csv"
COMPOSITION_CSV   = TRAIN_PKG_DIR / "streams_composition.csv"
TRAIN_STREAMS_DIR = TRAIN_PKG_DIR / "streams" / "train"
VAL_STREAMS_DIR   = TRAIN_PKG_DIR / "streams" / "val"
TEST_STREAMS_DIR  = TEST_PKG_DIR / "streams" / "test"

CKPT_DIR       = ROOT_DIR / "checkpoints_cascade"          # persistent (Drive)
CACHE_DIR      = (Path("/content/cache_cascade") if IN_COLAB else ROOT_DIR / "_cache_cascade")  # fast local disk
SUBMISSION_CSV = ROOT_DIR / "submission_colab.csv"
for _d in (CKPT_DIR, CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)


def ensure_file_from_zip(disk_path: Path, zip_path: Path, zip_member: str) -> Path:
    """Return disk_path if it exists; otherwise extract zip_member into CACHE_DIR and return that path."""
    if disk_path.is_file():
        return disk_path
    if zip_path.is_file():
        cached = CACHE_DIR / Path(zip_member).name
        if not cached.is_file():
            with zipfile.ZipFile(zip_path, "r") as zf:
                matches = [m for m in zf.namelist() if m == zip_member or m.endswith("/" + Path(zip_member).name)]
                if matches:
                    cached.write_bytes(zf.read(matches[0]))
        if cached.is_file():
            return cached
    return disk_path


EVENTS_CSV = ensure_file_from_zip(EVENTS_CSV, TRAIN_ZIP_PATH, "abc2027_uwb_train_v1/streams_events.csv")
COMPOSITION_CSV = ensure_file_from_zip(COMPOSITION_CSV, TRAIN_ZIP_PATH, "abc2027_uwb_train_v1/streams_composition.csv")

for _name, _p in [("TRAIN_PKG_DIR", TRAIN_PKG_DIR), ("TEST_PKG_DIR", TEST_PKG_DIR),
                  ("TRAIN_ZIP_PATH", TRAIN_ZIP_PATH), ("TEST_ZIP_PATH", TEST_ZIP_PATH),
                  ("EVENTS_CSV", EVENTS_CSV), ("COMPOSITION_CSV", COMPOSITION_CSV)]:
    print(f"[{'OK' if _p.exists() else 'MISSING'}] {_name:16s}: {_p}")


def stream_h5_path(stream_id: str) -> Path | None:
    """Resolve an extracted HDF5 path if present on disk; returns None when the stream lives inside a .zip."""
    split, idx = stream_id.split("_")
    base = {"train": TRAIN_STREAMS_DIR, "val": VAL_STREAMS_DIR, "test": TEST_STREAMS_DIR}[split]
    pkg = TEST_PKG_DIR if split == "test" else TRAIN_PKG_DIR
    candidates = [base / f"stream_{idx}.h5", base / f"{split}_{idx}.h5",
                  pkg / f"{split}_{idx}.h5", ROOT_DIR / f"{split}_{idx}.h5",
                  ROOT_DIR / "streams" / split / f"stream_{idx}.h5"]
    for c in candidates:
        if c.is_file():
            return c
    return None


def load_stream_cir(stream_id: str, h5_path: Path | None = None) -> np.ndarray:
    """Load `cir` (3, T, 368) float32 from an extracted .h5 file OR directly in-memory from the .zip archive."""
    disk = Path(h5_path) if h5_path is not None else stream_h5_path(stream_id)
    if disk is not None and disk.is_file():
        with h5py.File(disk, "r") as h:
            cir = np.nan_to_num(h["cir"][...].astype(np.float32, copy=False))
            serials = h["radar_serials"][:] if "radar_serials" in h else None
    else:
        split, idx = stream_id.split("_")
        zpath = TEST_ZIP_PATH if split == "test" else TRAIN_ZIP_PATH
        pkg_name = "abc2027_uwb_test_v1" if split == "test" else "abc2027_uwb_train_v1"
        if not zpath.is_file():
            raise FileNotFoundError(
                f"No HDF5 file on disk for {stream_id} and zip archive not found at {zpath}"
            )
        target_suffix = f"streams/{split}/stream_{idx}.h5"
        member = f"{pkg_name}/{target_suffix}"
        with zipfile.ZipFile(zpath, "r") as zf:
            names = zf.namelist()
            if member not in names:
                matches = [m for m in names if m.endswith(target_suffix) or m.endswith(f"{split}_{idx}.h5")]
                if not matches:
                    raise FileNotFoundError(f"Member {member} not found inside {zpath}")
                member = matches[0]
            with h5py.File(io.BytesIO(zf.read(member)), "r") as h:
                cir = np.nan_to_num(h["cir"][...].astype(np.float32, copy=False))
                serials = h["radar_serials"][:] if "radar_serials" in h else None

    if serials is not None and len(serials) == 3:
        order = np.argsort(serials)
        if not np.array_equal(order, np.arange(3)):
            cir = cir[order]
    return cir


# ----------------------------------------------------------------------------
# 3. Library imports for the official evaluator (optional, used for validation / format check)
# ----------------------------------------------------------------------------
if (TRAIN_PKG_DIR / "src").exists() and str(TRAIN_PKG_DIR / "src") not in sys.path:
    sys.path.insert(0, str(TRAIN_PKG_DIR / "src"))
try:
    from abc2027 import evaluate as abc_eval
except Exception as _e:                    # evaluator not available -> validation cell degrades gracefully
    abc_eval = None
    print("[WARN] abc2027.evaluate not importable:", _e)

# ----------------------------------------------------------------------------
# 4. Radar constants
# ----------------------------------------------------------------------------
FS_HZ      = 50.0
NUM_RADARS = 3
NUM_BINS   = 368

# 4 spatial bands per radar (inclusive bin ranges 0-65, 75-180, 183-225, 260-367) -> 12 channels
SPATIAL_BANDS = ((0, 66), (75, 181), (183, 226), (260, 368))      # python slice bounds
NUM_BANDS     = len(SPATIAL_BANDS)
NUM_CHANNELS  = NUM_RADARS * NUM_BANDS                             # 12

# ----------------------------------------------------------------------------
# 5. Device / reproducibility / dtype policy (strict float32 everywhere)
# ----------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float32
torch.set_default_dtype(torch.float32)
SEED = 1234
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
print("DEVICE:", DEVICE, "| torch", torch.__version__)

# ----------------------------------------------------------------------------
# 6. Pipeline hyper-parameters (single place to tune)
# ----------------------------------------------------------------------------
TRAIN_STREAM_IDS = [f"train_{i:02d}" for i in range(13)]
VAL_STREAM_IDS   = [f"val_{i:02d}" for i in range(5)]
TEST_STREAM_IDS  = [f"test_{i:02d}" for i in range(5)]
USE_VAL_FOR_TRAINING = False      # True -> final fit on train+val (val-tuned thresholds are then optimistic)
FORCE_RETRAIN        = False      # False -> reuse checkpoints found in CKPT_DIR

# Stage 1 (MobileNetV3 proposal)
S1_WINDOW_FRAMES = int(2.0 * FS_HZ)      # 100 frames = 2.0 s
S1_STRIDE_FRAMES = int(0.5 * FS_HZ)      # inference stride 0.5 s
S1_THRESHOLD     = 0.25
S1_STEPS, S1_BATCH, S1_LR = 3000, 64, 2e-3

# Stage 2 (1D-FPN decoder)
S2_STEPS, S2_BATCH, S2_LR = 4000, 16, 1e-3
S2_MIN_BLOCK_S, S2_MAX_BLOCK_S = 10.0, 15.0

# Inference / post-processing
P_FALL_THRESHOLD   = 0.5
SOFT_NMS_IOU       = 0.3
SOFT_NMS_FLOOR     = 0.3      # overwritten by the validation sweep in Cell 6C

## Cell 2 - Signal Preprocessing & Feature Engineering
* `SVDClutterRemoval(k=2)`: eigen-decomposition (`np.linalg.eigh`) of the Gram matrix `XᵀX` of the RTM, removal of the 2 dominant (static) components. Applied per 10-s block because streams are concatenations of different recordings.
* `extract_12_spatial_bands`: L2 norm of the residual over bins 0-65, 75-180, 183-225, 260-367 for 3 radars → `(12, Seq_Len)`.
* `RobustIQRNormalization`: `(x − median) / IQR` per channel.
* `preprocess_stream`: runs the above for a whole stream and caches the Stage-2 bands plus a float16 log-amplitude RTM (Stage-1 input) on local disk.

In [ ]:
class SVDClutterRemoval:
    """Static-clutter removal of a Range-Time Map (RTM) with a rank-k SVD.

    For an RTM X in R^{T x 368} the right singular vectors are the eigenvectors of the Gram matrix
    G = X^T X (368 x 368).  The k dominant eigenvectors V_k span the static background (furniture,
    walls) so the residual is  R = X - X V_k V_k^T.
    `process_stream` applies this independently on consecutive blocks (default 10 s) because the
    published streams are concatenations of different recordings with different clutter.
    """

    def __init__(self, k: int = 2, block_frames: int = 500):
        self.k = int(k)
        self.block_frames = int(block_frames)

    def __call__(self, rtm: np.ndarray) -> np.ndarray:
        """rtm: (T, 368) -> residual (T, 368), float32."""
        X = np.asarray(rtm, dtype=np.float64)
        gram = X.T @ X                                    # (368, 368), symmetric PSD
        _, eigvecs = np.linalg.eigh(gram)                 # eigenvalues ascending
        Vk = eigvecs[:, -self.k:]                         # k dominant (static) directions
        residual = X - (X @ Vk) @ Vk.T
        return residual.astype(np.float32)

    def _radar_blocks(self, T: int):
        B = self.block_frames
        edges = list(range(0, T, B)) + [T]
        if len(edges) > 2 and edges[-1] - edges[-2] < B // 2:   # merge a too-short tail block
            edges.pop(-2)
        return list(zip(edges[:-1], edges[1:]))

    def process_radar(self, rtm: np.ndarray, out: np.ndarray = None) -> np.ndarray:
        out = np.empty(rtm.shape, dtype=np.float32) if out is None else out
        for a, b in self._radar_blocks(rtm.shape[0]):
            out[a:b] = self(rtm[a:b])
        return out

    def process_stream(self, cir: np.ndarray, inplace: bool = True) -> np.ndarray:
        """cir: (3, T, 368) float32 -> residual (3, T, 368) float32.

        Radars are processed sequentially: LAPACK already multi-threads the 368x368 eigh and Python-level
        threads around it proved unstable on some platforms."""
        out = cir if inplace else np.empty_like(cir)
        for r in range(cir.shape[0]):
            self.process_radar(cir[r], out[r])
        return out


def extract_12_spatial_bands(residual_cir: np.ndarray) -> np.ndarray:
    """L2 norm over 4 range bands (bins 0-65, 75-180, 183-225, 260-367) for each of 3 radars.

    residual_cir: (3, Seq_Len, 368) -> (12, Seq_Len) float32, channel = radar * 4 + band.
    """
    assert residual_cir.ndim == 3 and residual_cir.shape[2] == NUM_BINS, residual_cir.shape
    R, T, _ = residual_cir.shape
    bands = np.empty((R * NUM_BANDS, T), dtype=np.float32)
    for r in range(R):
        for b, (lo, hi) in enumerate(SPATIAL_BANDS):
            seg = residual_cir[r, :, lo:hi].astype(np.float64, copy=False)
            bands[r * NUM_BANDS + b] = np.sqrt(np.einsum("tb,tb->t", seg, seg))
    return bands


class RobustIQRNormalization:
    """Per-channel robust scaling  (x - median) / IQR   with IQR = Q75 - Q25.

    Fitted once per stream (statistics are dominated by background, falls are rare), then applied to
    any slice of that stream, so training crops and inference blocks see identical scaling.
    """

    def __init__(self, eps: float = 1e-12, clip: float = 50.0):
        self.eps, self.clip = eps, clip
        self.median = None
        self.iqr = None

    def fit(self, x: np.ndarray):
        """x: (C, T)."""
        x64 = np.asarray(x, dtype=np.float64)
        q25, med, q75 = np.percentile(x64, [25, 50, 75], axis=1)
        iqr = q75 - q25
        fallback = x64.std(axis=1)                                   # degenerate (flat) channels
        self.iqr = np.where(iqr > self.eps, iqr, np.maximum(fallback, self.eps))
        self.median = med
        return self

    def transform(self, x: np.ndarray) -> np.ndarray:
        y = (np.asarray(x, dtype=np.float64) - self.median[:, None]) / self.iqr[:, None]
        if self.clip is not None:
            y = np.clip(y, -self.clip, self.clip)
        return y.astype(np.float32)

    def fit_transform(self, x: np.ndarray) -> np.ndarray:
        return self.fit(x).transform(x)


# ----------------------------------------------------------------------------
# Whole-stream preprocessing with an on-disk cache (bands for Stage 2, log-amplitude RTM for Stage 1)
# ----------------------------------------------------------------------------
class StreamData:
    """Container: bands (12, T) float32 | rtm (3, T, 368) float16 memmap | n_frames | duration."""

    def __init__(self, stream_id, bands, rtm):
        self.stream_id, self.bands, self.rtm = stream_id, bands, rtm
        self.n_frames = int(bands.shape[1])
        self.duration = self.n_frames / FS_HZ


def preprocess_stream(stream_id: str, h5_path: Path = None, cache_dir: Path = CACHE_DIR,
                      svd: SVDClutterRemoval = None, force: bool = False) -> StreamData:
    cache_dir = Path(cache_dir)
    bands_p, rtm_p = cache_dir / f"{stream_id}_bands.npy", cache_dir / f"{stream_id}_rtm.npy"
    if bands_p.exists() and rtm_p.exists() and not force:
        return StreamData(stream_id, np.load(bands_p), np.load(rtm_p, mmap_mode="r"))

    svd = svd or SVDClutterRemoval(k=2)
    t0 = time.time()
    cir = load_stream_cir(stream_id, h5_path=h5_path)                         # (3, T, 368) from disk or .zip
    assert cir.shape[0] == NUM_RADARS and cir.shape[2] == NUM_BINS, cir.shape
    residual = svd.process_stream(cir, inplace=True)                          # static clutter removed

    bands = extract_12_spatial_bands(residual)                                # (12, T)
    bands = RobustIQRNormalization().fit_transform(bands)                     # (x - median) / IQR

    # Stage-1 image: log1p(|residual| / robust per-radar scale) stored as float16 on disk
    T = residual.shape[1]
    rtm = np.lib.format.open_memmap(rtm_p, mode="w+", dtype=np.float16, shape=(NUM_RADARS, T, NUM_BINS))
    for r in range(NUM_RADARS):
        scale = float(np.median(np.abs(residual[r, ::25]))) + 1e-12
        for a in range(0, T, 20000):
            rtm[r, a:a + 20000] = np.log1p(np.abs(residual[r, a:a + 20000]) / scale).astype(np.float16)
    rtm.flush(); del rtm
    np.save(bands_p, bands)
    del residual, cir; gc.collect()
    print(f"  [preprocess] {stream_id}: T={T} ({T / FS_HZ:.0f}s) in {time.time() - t0:.0f}s")
    return StreamData(stream_id, bands, np.load(rtm_p, mmap_mode="r"))

## Cell 3 - Stage 1: Fast Proposal Network (MobileNetV3)
Pure-PyTorch MobileNetV3-Small (inverted residuals, squeeze-and-excitation, h-swish/h-sigmoid) on 2-s RTM chunks `(B, 3, 100, 368)` → fall probability.
`extract_suspicious_regions` binarises `p > 0.25`, merges nearby hits and pads them into 10-15 s blocks.

In [ ]:
class HSwish(nn.Module):
    """h-swish(x) = x * ReLU6(x + 3) / 6"""
    def forward(self, x):
        return x * F.relu6(x + 3.0) / 6.0


class HSigmoid(nn.Module):
    """h-sigmoid(x) = ReLU6(x + 3) / 6"""
    def forward(self, x):
        return F.relu6(x + 3.0) / 6.0


def _make_divisible(v: float, divisor: int = 8) -> int:
    new_v = max(divisor, int(v + divisor / 2) // divisor * divisor)
    return new_v + divisor if new_v < 0.9 * v else new_v


class SqueezeExcite(nn.Module):
    """Squeeze-and-Excitation:  x * h-sigmoid(W2 ReLU(W1 GAP(x)))   (reduction 4, MobileNetV3)."""
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        squeeze = _make_divisible(channels // reduction, 8)
        self.fc1 = nn.Conv2d(channels, squeeze, 1)
        self.fc2 = nn.Conv2d(squeeze, channels, 1)
        self.gate = HSigmoid()

    def forward(self, x):
        s = F.adaptive_avg_pool2d(x, 1)
        s = self.gate(self.fc2(F.relu(self.fc1(s), inplace=True)))
        return x * s


class InvertedResidualSE(nn.Module):
    """MobileNetV3 bneck: 1x1 expand -> kxk depthwise -> [SE] -> 1x1 linear projection (+ skip)."""
    def __init__(self, c_in, k, c_exp, c_out, use_se, use_hs, stride):
        super().__init__()
        act = HSwish if use_hs else (lambda: nn.ReLU(inplace=True))
        layers = []
        if c_exp != c_in:
            layers += [nn.Conv2d(c_in, c_exp, 1, bias=False), nn.BatchNorm2d(c_exp), act()]
        layers += [nn.Conv2d(c_exp, c_exp, k, stride, padding=k // 2, groups=c_exp, bias=False),
                   nn.BatchNorm2d(c_exp), act()]
        if use_se:
            layers.append(SqueezeExcite(c_exp))
        layers += [nn.Conv2d(c_exp, c_out, 1, bias=False), nn.BatchNorm2d(c_out)]
        self.block = nn.Sequential(*layers)
        self.use_skip = stride == 1 and c_in == c_out

    def forward(self, x):
        y = self.block(x)
        return x + y if self.use_skip else y


class MobileNetV3ProposalNet(nn.Module):
    """MobileNetV3-Small on 2-s RTM chunks.

    Input : (B, 3, 100, 368) float32 - 3 radars as channels, 100 frames (2.0 s) x 368 range bins.
    Output: (B,) probability that a fall overlaps the window (sigmoid); `logits()` gives the raw score.
    """
    # (kernel, expansion, out, SE, hard-swish, stride) - MobileNetV3-Small table (Howard et al., 2019)
    CFG = [(3, 16, 16, True, False, 2),
           (3, 72, 24, False, False, 2),
           (3, 88, 24, False, False, 1),
           (5, 96, 40, True, True, 2),
           (5, 240, 40, True, True, 1),
           (5, 240, 40, True, True, 1),
           (5, 120, 48, True, True, 1),
           (5, 144, 48, True, True, 1),
           (5, 288, 96, True, True, 2),
           (5, 576, 96, True, True, 1),
           (5, 576, 96, True, True, 1)]

    def __init__(self, in_channels: int = NUM_RADARS, dropout: float = 0.2):
        super().__init__()
        stem_c = 16
        layers = [nn.Conv2d(in_channels, stem_c, 3, 2, 1, bias=False), nn.BatchNorm2d(stem_c), HSwish()]
        c_in = stem_c
        for k, exp, out, se, hs, s in self.CFG:
            layers.append(InvertedResidualSE(c_in, k, exp, out, se, hs, s))
            c_in = out
        last_c = 6 * c_in                                                   # 576
        layers += [nn.Conv2d(c_in, last_c, 1, bias=False), nn.BatchNorm2d(last_c), HSwish()]
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Linear(last_c, 1024), HSwish(), nn.Dropout(dropout), nn.Linear(1024, 1))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.01); nn.init.zeros_(m.bias)

    def logits(self, x: torch.Tensor) -> torch.Tensor:
        z = F.adaptive_avg_pool2d(self.features(x), 1).flatten(1)
        return self.classifier(z).squeeze(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.logits(x))


@torch.no_grad()
def stage1_scan(model: nn.Module, rtm, window: int = S1_WINDOW_FRAMES, stride: int = S1_STRIDE_FRAMES,
                batch_size: int = 128):
    """Slide the 2-s window over a whole stream. rtm: (3, T, 368) array/memmap.

    Returns (probs (N,) float32, starts_frames (N,) int64)."""
    model.eval()
    T = rtm.shape[1]
    starts = np.arange(0, max(T - window, 0) + 1, stride, dtype=np.int64)
    if starts[-1] + window < T:
        starts = np.append(starts, T - window)
    probs = np.empty(len(starts), dtype=np.float32)
    for i in range(0, len(starts), batch_size):
        sl = starts[i:i + batch_size]
        x = np.stack([np.asarray(rtm[:, s:s + window, :]) for s in sl]).astype(np.float32)
        x = torch.from_numpy(x).to(device=DEVICE, dtype=torch.float32)
        probs[i:i + len(sl)] = model(x).float().cpu().numpy()
    return probs, starts


def extract_suspicious_regions(probs, window_starts_s=None, window_s: float = 2.0, stride_s: float = 0.5,
                               threshold: float = S1_THRESHOLD, merge_gap_s: float = 2.0,
                               margin_s: float = 3.0, min_block_s: float = S2_MIN_BLOCK_S,
                               max_block_s: float = S2_MAX_BLOCK_S, total_duration_s: float = None):
    """Binarise Stage-1 window probabilities (p > 0.25), merge nearby hits, pad to 10-15 s blocks.

    probs: (N,) window probabilities. window_starts_s: (N,) window start times [s]
           (default: i * stride_s).
    Returns a sorted list of (block_start_s, block_end_s) with 10 <= length <= 15 s.
    """
    probs = np.asarray(probs, dtype=np.float32).reshape(-1)
    starts = (np.arange(len(probs)) * stride_s if window_starts_s is None
              else np.asarray(window_starts_s, dtype=np.float64))
    hits = np.flatnonzero(probs > threshold)
    if hits.size == 0:
        return []

    merged = []                                           # 1) merge hits closer than merge_gap_s
    for i in hits:
        s, e = float(starts[i]), float(starts[i]) + window_s
        if merged and s - merged[-1][1] <= merge_gap_s:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])

    blocks = []                                           # 2) pad into [min_block_s, max_block_s]
    for s, e in merged:
        s, e = s - margin_s, e + margin_s
        if e - s < min_block_s:
            c = 0.5 * (s + e); s, e = c - 0.5 * min_block_s, c + 0.5 * min_block_s
        if e - s <= max_block_s:
            blocks.append([s, e])
        else:                                             # long activity -> overlapping 15-s tiles
            cur, step = s, max_block_s - margin_s
            while cur + max_block_s < e:
                blocks.append([cur, cur + max_block_s]); cur += step
            blocks.append([e - max_block_s, e])

    if total_duration_s is not None:                      # 3) clip to the stream, keep block length
        clipped = []
        for s, e in blocks:
            L = min(e - s, total_duration_s)
            if s < 0.0:
                s, e = 0.0, L
            if e > total_duration_s:
                s, e = total_duration_s - L, total_duration_s
            clipped.append([max(s, 0.0), e])
        blocks = clipped

    blocks.sort()                                         # 4) fuse overlapping blocks if still <= 15 s
    out = [blocks[0]]
    for s, e in blocks[1:]:
        if s <= out[-1][1] and e - out[-1][0] <= max_block_s:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return [(float(s), float(e)) for s, e in out]


# quick self-check
_net = MobileNetV3ProposalNet().to(DEVICE)
with torch.no_grad():
    _y = _net.eval()(torch.zeros(2, NUM_RADARS, S1_WINDOW_FRAMES, NUM_BINS, dtype=torch.float32, device=DEVICE))
print(f"MobileNetV3ProposalNet: {sum(p.numel() for p in _net.parameters()) / 1e6:.2f} M params, out {tuple(_y.shape)}")
del _net, _y

## Cell 4 - Stage 2: Frame-wise 1D-FPN Temporal Decoder (ActionFormer style)
Stem + 5 dilated residual blocks (d = 1, 2, 4, 8, 16; receptive field 377 frames ≈ 7.54 s), a top-down FPN over the block outputs and two per-frame heads:
`p_fall(t)` `(B, L)` (bias init −4.6) and `reg_bounds(t) = (Δs, Δe)` `(B, 2, L)` with a ReLU.

In [ ]:
class ChannelNorm(nn.Module):
    """LayerNorm over the channel axis of a (B, C, L) tensor, independently for every frame.

    Unlike GroupNorm/InstanceNorm it does not normalise over time, so the absolute motion energy of a
    block (what separates a fall from a quiet block) is preserved."""
    def __init__(self, channels: int):
        super().__init__()
        self.ln = nn.LayerNorm(channels)

    def forward(self, x):
        return self.ln(x.transpose(1, 2)).transpose(1, 2)


class DilatedResidualBlock(nn.Module):
    """x + Conv(k, d) -> Norm -> ReLU -> Dropout -> Conv(k, d) -> Norm,  followed by ReLU."""
    def __init__(self, ch: int, dilation: int, kernel: int = 7, dropout: float = 0.1):
        super().__init__()
        pad = dilation * (kernel - 1) // 2                       # 'same' length
        self.conv1 = nn.Conv1d(ch, ch, kernel, padding=pad, dilation=dilation)
        self.norm1 = ChannelNorm(ch)
        self.conv2 = nn.Conv1d(ch, ch, kernel, padding=pad, dilation=dilation)
        self.norm2 = ChannelNorm(ch)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y = self.drop(F.relu(self.norm1(self.conv1(x))))
        y = self.norm2(self.conv2(y))
        return F.relu(x + y)


class TemporalFPNDecoder(nn.Module):
    """Frame-wise 1D-FPN temporal decoder (ActionFormer-style heads).

    x: (B, 12, L)  ->  p_fall (B, L) in (0,1)  and  reg_bounds (B, 2, L) = (delta_start, delta_end) >= 0 [s]

    Backbone: Conv1d stem (k=5) + 5 dilated residual blocks (d = 1, 2, 4, 8, 16, two k=7 convs each)
              receptive field = 1 + 4 + 2 * 6 * (1+2+4+8+16) = 377 frames = 7.54 s.
    FPN     : the 5 block outputs are the pyramid levels (increasing temporal context). A top-down
              pathway P_i = smooth(lateral(C_i) + P_{i+1}) injects wide-context features into the
              fine levels, and all levels are fused into one per-frame feature (full 50 Hz resolution).
    Heads   : classification (bias init -4.6 -> p0 ~ 0.01) and regression (final ReLU, distances >= 0).
    """

    def __init__(self, in_channels: int = NUM_CHANNELS, channels: int = 64,
                 dilations=(1, 2, 4, 8, 16), kernel: int = 7, dropout: float = 0.1):
        super().__init__()
        self.dilations, self.kernel = tuple(dilations), kernel
        self.stem = nn.Sequential(nn.Conv1d(in_channels, channels, 5, padding=2), ChannelNorm(channels), nn.ReLU())
        self.blocks = nn.ModuleList([DilatedResidualBlock(channels, d, kernel, dropout) for d in self.dilations])
        n = len(self.dilations)
        self.lateral = nn.ModuleList([nn.Conv1d(channels, channels, 1) for _ in range(n)])
        self.smooth = nn.ModuleList([nn.Conv1d(channels, channels, 3, padding=1) for _ in range(n)])
        self.fuse = nn.Sequential(nn.Conv1d(n * channels, channels, 1), ChannelNorm(channels), nn.ReLU())

        self.cls_head = nn.Sequential(nn.Conv1d(channels, channels, 3, padding=1), nn.ReLU(),
                                      nn.Conv1d(channels, 1, 1))
        self.reg_head = nn.Sequential(nn.Conv1d(channels, channels, 3, padding=1), nn.ReLU(),
                                      nn.Conv1d(channels, 2, 1))
        nn.init.constant_(self.cls_head[-1].bias, -4.6)           # focal-loss prior: p(fall) ~ 0.01
        nn.init.normal_(self.cls_head[-1].weight, 0.0, 0.01)
        nn.init.normal_(self.reg_head[-1].weight, 0.0, 0.01)
        nn.init.constant_(self.reg_head[-1].bias, 1.0)            # ~1 s so the output ReLU starts alive

    @property
    def receptive_field_frames(self) -> int:
        return 1 + 4 + 2 * (self.kernel - 1) * sum(self.dilations)

    def forward_logits(self, x: torch.Tensor):
        h = self.stem(x)
        feats = []
        for blk in self.blocks:
            h = blk(h)
            feats.append(h)
        top, pyramid = None, [None] * len(feats)
        for i in reversed(range(len(feats))):                     # top-down pathway
            lat = self.lateral[i](feats[i])
            top = self.smooth[i](lat if top is None else lat + top)
            pyramid[i] = top
        z = self.fuse(torch.cat(pyramid, dim=1))
        cls_logit = self.cls_head(z).squeeze(1)                   # (B, L)
        reg = F.relu(self.reg_head(z))                            # (B, 2, L), distances >= 0
        return cls_logit, reg

    def forward(self, x: torch.Tensor):
        cls_logit, reg = self.forward_logits(x)
        return torch.sigmoid(cls_logit), reg


# quick self-check
_dec = TemporalFPNDecoder().to(DEVICE)
with torch.no_grad():
    _p, _r = _dec.eval()(torch.zeros(2, NUM_CHANNELS, 600, dtype=torch.float32, device=DEVICE))
print(f"TemporalFPNDecoder: {sum(p.numel() for p in _dec.parameters()) / 1e6:.2f} M params | "
      f"p_fall {tuple(_p.shape)} reg {tuple(_r.shape)} | receptive field {_dec.receptive_field_frames} frames "
      f"= {_dec.receptive_field_frames / FS_HZ:.2f} s | init p ~ {_p.mean().item():.4f}")
del _dec, _p, _r

## Cell 5 - Ground-Truth Generator & Temporal DIoU Loss
`Total = Focal(α=0.85, γ=2) + Splice_Penalty (5× on false positives in 0.2-s crossfade junctions) + 2.0 · DIoU`,
with `DIoU = 1 − (IoU − CenterDist² / EnclosingWidth²)` computed only where `cls_target == 1`.

In [ ]:
def create_regression_targets(gt_events, n_frames: int, fs: float = FS_HZ):
    """Frame-wise targets for one block.

    gt_events: iterable of (t_start, t_end) [s] (or dicts with those keys) relative to the block start.
    Returns cls_target (n_frames,) and reg_target (2, n_frames), both float32 tensors:
      inside [t_start, t_end] : cls = 1, reg[0] = t - t_start, reg[1] = t_end - t   (seconds)
      elsewhere               : cls = 0, reg = 0
    """
    t = torch.arange(n_frames, dtype=torch.float32) / fs
    cls_target = torch.zeros(n_frames, dtype=torch.float32)
    reg_target = torch.zeros(2, n_frames, dtype=torch.float32)
    for ev in gt_events:
        ts, te = (ev["t_start"], ev["t_end"]) if isinstance(ev, dict) else (ev[0], ev[1])
        inside = (t >= ts) & (t <= te)
        cls_target[inside] = 1.0
        reg_target[0, inside] = t[inside] - float(ts)
        reg_target[1, inside] = float(te) - t[inside]
    return cls_target, reg_target


def build_splice_mask(composition_df: pd.DataFrame, stream_id: str, n_frames: int,
                      fs: float = FS_HZ, crossfade_s: float = 0.2, pad_s: float = 0.1) -> np.ndarray:
    """Boolean (n_frames,) mask of the 0.2-s crossfade zones where consecutive source segments are joined
    (segment k starts at its t_start; the cross-fade covers [t_start, t_start + 0.2 s]), +-pad_s margin."""
    mask = np.zeros(n_frames, dtype=bool)
    seg = composition_df[composition_df["stream_id"] == stream_id].sort_values("t_start")
    for t_start in seg["t_start"].to_numpy()[1:]:                   # the first segment has no junction
        a = max(int(round((t_start - pad_s) * fs)), 0)
        b = min(int(round((t_start + crossfade_s + pad_s) * fs)) + 1, n_frames)
        if b > a:
            mask[a:b] = True
    return mask


def focal_loss_map(p: torch.Tensor, target: torch.Tensor, alpha: float, gamma: float, eps: float = 1e-6):
    """Per-frame focal loss  FL = -alpha_t (1 - p_t)^gamma log(p_t)."""
    p = p.clamp(eps, 1.0 - eps)
    pos = target > 0.5
    p_t = torch.where(pos, p, 1.0 - p)
    alpha_t = torch.where(pos, torch.full_like(p, alpha), torch.full_like(p, 1.0 - alpha))
    return -alpha_t * (1.0 - p_t).pow(gamma) * torch.log(p_t)


def temporal_diou(pred_bounds: torch.Tensor, tgt_bounds: torch.Tensor, t: torch.Tensor, eps: float = 1e-6):
    """1-D Distance-IoU for frames at times t with (delta_start, delta_end) distances.

    pred interval [t - ds, t + de]  vs  GT interval [t - gs, t + ge];  all tensors (N,) / (N, 2).
      IoU  = I / U,   I = max(0, min(pe, ge) - max(ps, gs)),   U = (pe - ps) + (ge - gs) - I
      DIoU = IoU - (c_p - c_g)^2 / W^2        (W = width of the smallest enclosing interval)
    returns (1 - DIoU, IoU)."""
    ps, pe = t - pred_bounds[:, 0], t + pred_bounds[:, 1]
    gs, ge = t - tgt_bounds[:, 0], t + tgt_bounds[:, 1]
    inter = (torch.minimum(pe, ge) - torch.maximum(ps, gs)).clamp(min=0.0)
    union = (pe - ps) + (ge - gs) - inter
    iou = inter / (union + eps)
    center_dist2 = (0.5 * (ps + pe) - 0.5 * (gs + ge)).pow(2)
    enclose_w = torch.maximum(pe, ge) - torch.minimum(ps, gs)
    diou = iou - center_dist2 / (enclose_w.pow(2) + eps)
    return 1.0 - diou, iou


class TemporalDIoULoss(nn.Module):
    """Total = Focal(alpha=0.85, gamma=2) + Splice_Penalty + 2.0 * DIoU

    * Focal      : all frames, normalised by the number of positive frames (>= 1).
    * Splice     : splice_weight (5) x focal loss of *negative* frames inside crossfade junction zones
                   (added on top of the plain focal term), same normalisation -> punishes FPs at splices.
    * DIoU       : only frames with cls_target == 1, mean over those frames.
    """

    def __init__(self, alpha: float = 0.85, gamma: float = 2.0, splice_weight: float = 5.0,
                 diou_weight: float = 2.0, fs: float = FS_HZ):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
        self.splice_weight, self.diou_weight, self.fs = splice_weight, diou_weight, fs

    def forward(self, p_fall, reg_bounds, cls_target, reg_target, splice_mask=None):
        """p_fall (B,L) | reg_bounds (B,2,L) | cls_target (B,L) | reg_target (B,2,L) | splice_mask (B,L) bool."""
        p_fall, reg_bounds = p_fall.to(torch.float32), reg_bounds.to(torch.float32)
        pos = cls_target > 0.5
        n_pos = pos.sum().clamp(min=1).to(torch.float32)

        fl = focal_loss_map(p_fall, cls_target, self.alpha, self.gamma)
        focal = fl.sum() / n_pos

        if splice_mask is not None:
            junction_neg = splice_mask.to(torch.bool) & (~pos)
            splice = self.splice_weight * (fl * junction_neg.to(torch.float32)).sum() / n_pos
        else:
            splice = fl.new_zeros(())

        if pos.any():
            B, L = p_fall.shape
            t = (torch.arange(L, device=p_fall.device, dtype=torch.float32) / self.fs).expand(B, L)
            pred = reg_bounds.permute(0, 2, 1)[pos]               # (N, 2)
            tgt = reg_target.permute(0, 2, 1)[pos]                # (N, 2)
            diou_l, iou = temporal_diou(pred, tgt, t[pos])
            diou_loss, mean_iou = diou_l.mean(), iou.mean().detach()
        else:
            diou_loss, mean_iou = reg_bounds.sum() * 0.0, reg_bounds.new_zeros(())

        total = focal + splice + self.diou_weight * diou_loss
        return total, {"focal": float(focal.detach()), "splice": float(splice.detach()),
                       "diou": float(diou_loss.detach()), "iou": float(mean_iou), "n_pos": int(n_pos)}


# ---- unit checks of the exact math ----
_c, _r = create_regression_targets([(1.0, 2.0)], 150)
assert _c.dtype == torch.float32 and _r.dtype == torch.float32
assert int(_c.sum()) == 51 and abs(float(_r[0, 75]) - 0.5) < 1e-6 and abs(float(_r[1, 75]) - 0.5) < 1e-6
_t = torch.tensor([1.5]); _g = torch.tensor([[0.5, 0.5]])
_l, _i = temporal_diou(torch.tensor([[0.5, 0.5]]), _g, _t); assert abs(float(_l)) < 1e-4 and abs(float(_i) - 1) < 1e-4
_l, _i = temporal_diou(torch.tensor([[0.25, 0.25]]), _g, _t); assert abs(float(_i) - 0.5) < 1e-4     # IoU = 0.5/1.0
_l, _i = temporal_diou(torch.tensor([[0.0, 0.5]]), _g, _t)   # pred [1.5,2.0] vs GT [1.0,2.0]: IoU .5, d=.25, W=1
assert abs(float(_l) - (1 - (0.5 - 0.0625))) < 1e-4, float(_l)
print("Loss unit checks passed.")

## Cell 6 - Inference Engine & 1D Soft-NMS
`decode_and_nms(p_fall, reg_bounds, t_offset)`: keep `p_fall > 0.5`, `start = t_offset + t − Δs`, `end = t_offset + t + Δe`, linear Soft-NMS (IoU threshold 0.3) → `{stream_id, start_time, end_time, score}`.

In [ ]:
def _iou_1d(box: np.ndarray, boxes: np.ndarray) -> np.ndarray:
    inter = np.clip(np.minimum(box[1], boxes[:, 1]) - np.maximum(box[0], boxes[:, 0]), 0.0, None)
    union = (box[1] - box[0]) + (boxes[:, 1] - boxes[:, 0]) - inter
    return inter / np.maximum(union, 1e-9)


def soft_nms_1d(boxes: np.ndarray, scores: np.ndarray, iou_thr: float = SOFT_NMS_IOU, method: str = "linear",
                sigma: float = 0.5, score_floor: float = None):
    """1-D Soft-NMS (Bodla et al.). boxes (N,2) [start,end], scores (N,).

    Repeatedly keeps the highest-scoring box and decays the score of the remaining boxes:
      linear  : s <- s * (1 - IoU)            if IoU > iou_thr
      gaussian: s <- s * exp(-IoU^2 / sigma)  for every box
    Boxes whose decayed score falls below `score_floor` are dropped. Returns (kept_boxes, kept_scores)."""
    score_floor = SOFT_NMS_FLOOR if score_floor is None else score_floor
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 2).copy()
    scores = np.asarray(scores, dtype=np.float64).reshape(-1).copy()
    kept_b, kept_s = [], []
    while len(scores) > 0:
        m = int(np.argmax(scores))
        best_b, best_s = boxes[m].copy(), float(scores[m])
        kept_b.append(best_b); kept_s.append(best_s)
        boxes, scores = np.delete(boxes, m, axis=0), np.delete(scores, m)
        if len(scores) == 0:
            break
        iou = _iou_1d(best_b, boxes)
        if method == "gaussian":
            decay = np.exp(-(iou ** 2) / sigma)
        else:
            decay = np.where(iou > iou_thr, 1.0 - iou, 1.0)
        scores = scores * decay
        alive = scores >= score_floor
        boxes, scores = boxes[alive], scores[alive]
    if not kept_b:
        return np.zeros((0, 2)), np.zeros(0)
    return np.stack(kept_b), np.asarray(kept_s)


@torch.no_grad()
def decode_and_nms(p_fall, reg_bounds, t_offset: float, stream_id: str = "stream", fs: float = FS_HZ,
                   score_thresh: float = P_FALL_THRESHOLD, iou_thr: float = SOFT_NMS_IOU,
                   score_floor: float = None, method: str = "linear", min_duration_s: float = 0.2,
                   t_max: float = None):
    """Frame-wise predictions of one block -> list of {stream_id, start_time, end_time, score}.

    p_fall (L,) | reg_bounds (2, L) [seconds] | t_offset = absolute time [s] of the block's first frame.
      keep frames with p_fall > score_thresh;  start = t_offset + t - ds,  end = t_offset + t + de;
      then 1-D Soft-NMS (IoU threshold 0.3)."""
    p = torch.as_tensor(p_fall).detach().to("cpu", torch.float32).reshape(-1).numpy()
    reg = torch.as_tensor(reg_bounds).detach().to("cpu", torch.float32).reshape(2, -1).numpy()
    t = np.arange(p.shape[0], dtype=np.float64) / fs
    keep = p > score_thresh
    if not keep.any():
        return []
    start = t_offset + t[keep] - reg[0, keep]
    end = t_offset + t[keep] + reg[1, keep]
    start = np.maximum(start, 0.0)
    if t_max is not None:
        end = np.minimum(end, t_max)
    valid = (end - start) >= min_duration_s
    if not valid.any():
        return []
    boxes = np.stack([start[valid], end[valid]], axis=1)
    kb, ks = soft_nms_1d(boxes, p[keep][valid], iou_thr, method, score_floor=score_floor)
    return [{"stream_id": stream_id, "start_time": float(b[0]), "end_time": float(b[1]), "score": float(s)}
            for b, s in zip(kb, ks)]


def merge_detections(dets, iou_thr: float = SOFT_NMS_IOU, score_floor: float = None, method: str = "linear"):
    """Soft-NMS across the detections of all blocks of a stream (blocks may overlap)."""
    if not dets:
        return []
    boxes = np.array([[d["start_time"], d["end_time"]] for d in dets])
    scores = np.array([d["score"] for d in dets])
    kb, ks = soft_nms_1d(boxes, scores, iou_thr, method, score_floor=score_floor)
    sid = dets[0]["stream_id"]
    out = [{"stream_id": sid, "start_time": float(b[0]), "end_time": float(b[1]), "score": float(s)}
           for b, s in zip(kb, ks)]
    return sorted(out, key=lambda d: d["start_time"])


# ----------------------------------------------------------------------------
# Cascade runner:  Stage 1 proposals -> Stage 2 frame-wise decoding (raw outputs are kept so that the
# post-processing thresholds can be swept cheaply on validation data).
# ----------------------------------------------------------------------------
@torch.no_grad()
def run_cascade(sd: StreamData, stage1: nn.Module, stage2: nn.Module):
    stage1.eval(); stage2.eval()
    probs, starts = stage1_scan(stage1, sd.rtm)
    regions = extract_suspicious_regions(probs, starts / FS_HZ, window_s=S1_WINDOW_FRAMES / FS_HZ,
                                         total_duration_s=sd.duration)
    outputs = []
    for (a_s, b_s) in regions:
        a, b = int(round(a_s * FS_HZ)), min(int(round(b_s * FS_HZ)), sd.n_frames)
        if b - a < 50:
            continue
        x = torch.from_numpy(np.ascontiguousarray(sd.bands[:, a:b])).to(device=DEVICE, dtype=torch.float32).unsqueeze(0)
        p, reg = stage2(x)
        outputs.append({"t_offset": a / FS_HZ, "p": p[0].float().cpu(), "reg": reg[0].float().cpu()})
    return {"s1_probs": probs, "s1_starts": starts, "regions": regions, "blocks": outputs}


def detections_from_outputs(result: dict, stream_id: str, duration: float, score_floor: float = None,
                            score_thresh: float = P_FALL_THRESHOLD):
    dets = []
    for blk in result["blocks"]:
        dets += decode_and_nms(blk["p"], blk["reg"], blk["t_offset"], stream_id, score_thresh=score_thresh,
                               score_floor=score_floor, t_max=duration)
    return merge_detections(dets, score_floor=score_floor)


# ---- unit check: two overlapping duplicates + one distinct event ----
_kb, _ks = soft_nms_1d(np.array([[1.0, 3.0], [1.1, 3.1], [10.0, 12.0]]), np.array([0.9, 0.8, 0.7]), 0.3, score_floor=0.3)
assert len(_kb) == 2 and abs(_ks[0] - 0.9) < 1e-9, (_kb, _ks)
print("Soft-NMS unit check passed.")

## Cell 6B - Training (Stage 1 then Stage 2)
Needed to produce weights: Stage 1 is trained with BCE on balanced 2-s windows (positives overlap a fall ≥ 0.5 s, 40 % hard negatives); Stage 2 with `TemporalDIoULoss` on random 10-15 s blocks (70 % contain a fall, 15 % hard negatives, 15 % random), splice masks from `streams_composition.csv`.
Checkpoints go to `ABC27/checkpoints_cascade/` and are re-used on the next run (`FORCE_RETRAIN = True` to retrain).

In [ ]:
EVENTS_DF = pd.read_csv(EVENTS_CSV)
COMPOSITION_DF = pd.read_csv(COMPOSITION_CSV)


class LabeledStream:
    """Preprocessed stream + fall events + splice-junction mask + hard-negative candidates."""

    def __init__(self, stream_id: str):
        self.sd = preprocess_stream(stream_id)
        self.stream_id, self.n_frames, self.bands, self.rtm = stream_id, self.sd.n_frames, self.sd.bands, self.sd.rtm
        ev = EVENTS_DF[(EVENTS_DF["stream_id"] == stream_id)]
        self.events = ev[["t_start", "t_end"]].to_numpy(dtype=np.float64)          # (E, 2) seconds
        self.splice = build_splice_mask(COMPOSITION_DF, stream_id, self.n_frames)
        # hard negatives: 2-s windows (stride 0.5 s) with the highest motion energy and no fall nearby
        energy = np.log1p(np.clip(self.bands, 0, None)).mean(0)
        csum = np.concatenate([[0.0], np.cumsum(energy, dtype=np.float64)])
        W, S = S1_WINDOW_FRAMES, S1_STRIDE_FRAMES
        starts = np.arange(0, self.n_frames - W, S)
        win_e = (csum[starts + W] - csum[starts]) / W
        ok = np.array([self.overlap_s(s, W) <= 0.0 for s in starts])
        cand = starts[ok & (win_e >= np.quantile(win_e, 0.85))]
        self.hard_starts = cand if len(cand) else starts

    def overlap_s(self, start_frame: int, length: int) -> float:
        """Max overlap [s] between [start, start+length) and any fall event."""
        if len(self.events) == 0:
            return 0.0
        a, b = start_frame / FS_HZ, (start_frame + length) / FS_HZ
        ov = np.minimum(b, self.events[:, 1]) - np.maximum(a, self.events[:, 0])
        return float(max(ov.max(), 0.0))


# ----------------------------------------------------------------------------
# Stage 1 sampler / trainer
# ----------------------------------------------------------------------------
def sample_stage1_batch(stores, batch_size: int, rng: np.random.Generator):
    W = S1_WINDOW_FRAMES
    xs, ys = [], []
    weights = np.array([max(len(s.events), 1) for s in stores], dtype=np.float64); weights /= weights.sum()
    while len(xs) < batch_size:
        positive = len(xs) % 2 == 0                                  # balanced 1:1
        st = stores[rng.choice(len(stores), p=weights)] if positive else stores[rng.integers(len(stores))]
        if positive and len(st.events):
            ts, te = st.events[rng.integers(len(st.events))]
            min_ov = min(0.5, 0.5 * (te - ts))                       # window must contain >= 0.5 s of the fall
            lo, hi = ts + min_ov - W / FS_HZ, te - min_ov
            s = int(round(rng.uniform(lo, hi) * FS_HZ))
        elif (not positive) and rng.random() < 0.4:
            s = int(rng.choice(st.hard_starts)) + int(rng.integers(-10, 11))
        else:
            s = int(rng.integers(0, st.n_frames - W))
        s = int(np.clip(s, 0, st.n_frames - W))
        ov = st.overlap_s(s, W)
        if positive and ov < 0.25:
            continue
        if (not positive) and ov > 0.0:
            continue
        xs.append(np.asarray(st.rtm[:, s:s + W, :], dtype=np.float32)); ys.append(1.0 if positive else 0.0)
    x = torch.from_numpy(np.stack(xs)).to(device=DEVICE, dtype=torch.float32)
    y = torch.tensor(ys, dtype=torch.float32, device=DEVICE)
    return x, y


def train_stage1(stores, steps=S1_STEPS, batch_size=S1_BATCH, lr=S1_LR, log_every=250):
    model = MobileNetV3ProposalNet().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    rng = np.random.default_rng(SEED)
    model.train(); run, t0 = 0.0, time.time()
    for it in range(1, steps + 1):
        x, y = sample_stage1_batch(stores, batch_size, rng)
        if rng.random() < 0.5:                                        # light augmentation: gain jitter
            x = x * float(rng.uniform(0.9, 1.1))
        loss = F.binary_cross_entropy_with_logits(model.logits(x), y)
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step(); sched.step()
        run = loss.item() if it == 1 else 0.98 * run + 0.02 * loss.item()
        if it % log_every == 0 or it == steps:
            print(f"  [Stage1] step {it:5d}/{steps}  bce {run:.4f}  ({time.time() - t0:.0f}s)")
    return model.eval()


@torch.no_grad()
def stage1_recall(model, stores):
    """Fraction of falls covered by at least one hit window (p > 0.25) + fraction of the stream flagged."""
    hit_ev = tot_ev = 0; flagged = total = 0.0
    for st in stores:
        probs, starts = stage1_scan(model, st.rtm)
        regions = extract_suspicious_regions(probs, starts / FS_HZ, window_s=S1_WINDOW_FRAMES / FS_HZ,
                                             total_duration_s=st.sd.duration)
        for ts, te in st.events:
            tot_ev += 1
            hit_ev += any(a <= (ts + te) / 2 <= b for a, b in regions)
        flagged += sum(b - a for a, b in regions); total += st.sd.duration
    return hit_ev / max(tot_ev, 1), flagged / max(total, 1e-9)


# ----------------------------------------------------------------------------
# Stage 2 sampler / trainer
# ----------------------------------------------------------------------------
def sample_stage2_batch(stores, batch_size: int, rng: np.random.Generator):
    L = int(rng.integers(int(S2_MIN_BLOCK_S * FS_HZ), int(S2_MAX_BLOCK_S * FS_HZ) + 1))   # one length per batch
    xs, cls_l, reg_l, spl_l = [], [], [], []
    weights = np.array([max(len(s.events), 1) for s in stores], dtype=np.float64); weights /= weights.sum()
    for _ in range(batch_size):
        r = rng.random()
        st = stores[rng.choice(len(stores), p=weights)] if r < 0.7 else stores[rng.integers(len(stores))]
        if r < 0.7 and len(st.events):                                # block containing a fall (random position)
            ts, te = st.events[rng.integers(len(st.events))]
            lo = int(te * FS_HZ) - L + 25
            hi = int(ts * FS_HZ) - 25
            start = int(rng.integers(lo, hi + 1)) if hi >= lo else int(ts * FS_HZ) - L // 2
        elif r < 0.85:                                                # hard negative (high-energy, fall-free)
            start = int(rng.choice(st.hard_starts)) - int(rng.integers(0, L))
        else:                                                         # random block
            start = int(rng.integers(0, st.n_frames - L + 1))
        start = int(np.clip(start, 0, st.n_frames - L))
        a, b = start, start + L
        x = st.bands[:, a:b].copy()
        x *= rng.uniform(0.85, 1.15)                                  # gain jitter
        x += rng.normal(0.0, 0.02, size=x.shape).astype(np.float32)
        t0 = start / FS_HZ
        evs = [(ts - t0, te - t0) for ts, te in st.events if te > t0 and ts < t0 + L / FS_HZ]
        c, rg = create_regression_targets(evs, L)
        xs.append(torch.from_numpy(x)); cls_l.append(c); reg_l.append(rg)
        spl_l.append(torch.from_numpy(st.splice[a:b].copy()))
    to = lambda t: torch.stack(t).to(DEVICE)
    return (to(xs).to(torch.float32), to(cls_l).to(torch.float32), to(reg_l).to(torch.float32), to(spl_l))


def train_stage2(stores, steps=S2_STEPS, batch_size=S2_BATCH, lr=S2_LR, log_every=250):
    model = TemporalFPNDecoder().to(DEVICE)
    criterion = TemporalDIoULoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    rng = np.random.default_rng(SEED + 1)
    model.train(); t0 = time.time(); agg = {}
    for it in range(1, steps + 1):
        x, cls_t, reg_t, spl = sample_stage2_batch(stores, batch_size, rng)
        p, reg = model(x)
        loss, parts = criterion(p, reg, cls_t, reg_t, spl)
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step(); sched.step()
        parts["total"] = float(loss.detach())
        for k, v in parts.items():
            agg[k] = v if k not in agg else 0.98 * agg[k] + 0.02 * v
        if it % log_every == 0 or it == steps:
            print(f"  [Stage2] step {it:5d}/{steps}  total {agg['total']:.3f}  focal {agg['focal']:.3f}  "
                  f"splice {agg['splice']:.3f}  diou {agg['diou']:.3f}  mIoU {agg['iou']:.3f}  ({time.time() - t0:.0f}s)")
    return model.eval()


# ----------------------------------------------------------------------------
# Run (or resume from checkpoints)
# ----------------------------------------------------------------------------
S1_CKPT, S2_CKPT = CKPT_DIR / "stage1_mobilenetv3.pt", CKPT_DIR / "stage2_fpn_decoder.pt"
need_train = FORCE_RETRAIN or not (S1_CKPT.exists() and S2_CKPT.exists())

stage1_model = MobileNetV3ProposalNet().to(DEVICE)
stage2_model = TemporalFPNDecoder().to(DEVICE)

if need_train:
    fit_ids = TRAIN_STREAM_IDS + (VAL_STREAM_IDS if USE_VAL_FOR_TRAINING else [])
    print(f"Preprocessing {len(fit_ids)} streams (SVD clutter removal + band features, cached in {CACHE_DIR}) ...")
    train_stores = [LabeledStream(s) for s in fit_ids]
    print(f"Training streams: {len(train_stores)}, fall events: {sum(len(s.events) for s in train_stores)}")
    stage1_model = train_stage1(train_stores)
    torch.save(stage1_model.state_dict(), S1_CKPT)
    stage2_model = train_stage2(train_stores)
    torch.save(stage2_model.state_dict(), S2_CKPT)
    del train_stores; gc.collect()
else:
    stage1_model.load_state_dict(torch.load(S1_CKPT, map_location=DEVICE))
    stage2_model.load_state_dict(torch.load(S2_CKPT, map_location=DEVICE))
    print("Loaded checkpoints from", CKPT_DIR)
stage1_model.eval(); stage2_model.eval()

## Cell 6C - Validation (optional but recommended)
Runs the full cascade on `val_00 … val_04`, reports Stage-1 recall, sweeps the Soft-NMS score floor and prints the official evaluator report.

In [ ]:
def _tiou(a0, a1, b0, b1):
    ov = min(a1, b1) - max(a0, b0)
    return 0.0 if ov <= 0 else ov / ((a1 - a0) + (b1 - b0) - ov)


def event_f1(gt_by_stream: dict, preds: list, thetas=(0.1, 0.3, 0.5)):
    """Event-level F1 (greedy one-to-one matching by descending score, same rule as the official scorer)."""
    out = {}
    for th in thetas:
        tp = fp = fn = 0
        for sid, gts in gt_by_stream.items():
            ps = sorted([p for p in preds if p["stream_id"] == sid], key=lambda p: -p["score"])
            used = set()
            for p in ps:
                best, best_iou = None, -1.0
                for j, (g0, g1) in enumerate(gts):
                    if j in used:
                        continue
                    iou = _tiou(p["t_start"], p["t_end"], g0, g1)
                    if iou >= th and iou > best_iou:
                        best, best_iou = j, iou
                if best is None:
                    fp += 1
                else:
                    used.add(best); tp += 1
            fn += len(gts) - len(used)
        prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
        out[th] = 2 * prec * rec / max(prec + rec, 1e-9)
    return out


def to_records(dets):
    return [{"stream_id": d["stream_id"], "t_start": d["start_time"], "t_end": d["end_time"],
             "score": d["score"], "t_alarm": None} for d in dets]


# ---- run the cascade once per validation stream and keep the raw frame-wise outputs ----
val_results, val_durations, val_gt = {}, {}, {}
for sid in VAL_STREAM_IDS:
    st = LabeledStream(sid)
    val_results[sid] = run_cascade(st.sd, stage1_model, stage2_model)
    val_durations[sid] = st.sd.duration
    val_gt[sid] = [tuple(e) for e in st.events]
    regs = val_results[sid]["regions"]
    cover = np.mean([any(a <= (g0 + g1) / 2 <= b for a, b in regs) for g0, g1 in val_gt[sid]]) if val_gt[sid] else float("nan")
    print(f"{sid}: {len(regs)} suspicious blocks ({sum(b - a for a, b in regs) / st.sd.duration:.0%} of stream), "
          f"Stage-1 fall recall {cover:.1%}")
    del st; gc.collect()

# ---- sweep the Soft-NMS score floor on validation (mean F1 @ 0.1 / 0.3 / 0.5 = ranking score) ----
sweep = {}
for floor in (0.05, 0.1, 0.2, 0.3, 0.4, 0.5):
    preds = []
    for sid in VAL_STREAM_IDS:
        preds += to_records(detections_from_outputs(val_results[sid], sid, val_durations[sid], score_floor=floor))
    f1 = event_f1(val_gt, preds)
    sweep[floor] = (float(np.mean(list(f1.values()))), f1, len(preds))
    print(f"  soft-NMS floor {floor:.2f}: #pred {len(preds):4d}  F1@0.1/0.3/0.5 = "
          f"{f1[0.1]:.3f}/{f1[0.3]:.3f}/{f1[0.5]:.3f}  mean {sweep[floor][0]:.3f}")
SOFT_NMS_FLOOR = max(sweep, key=lambda k: (round(sweep[k][0], 4), k))     # ties -> larger floor (fewer duplicates)
print(f"==> SOFT_NMS_FLOOR = {SOFT_NMS_FLOOR} (val ranking score {sweep[SOFT_NMS_FLOOR][0]:.3f})")

# ---- official evaluator report (if the package is available) ----
if abc_eval is not None:
    val_preds = []
    for sid in VAL_STREAM_IDS:
        val_preds += to_records(detections_from_outputs(val_results[sid], sid, val_durations[sid], score_floor=SOFT_NMS_FLOOR))
    gt_all = [e for e in abc_eval.load_ground_truth(EVENTS_CSV, ignore_flagged=True) if e["stream_id"] in val_durations]
    rep = abc_eval.evaluate(gt_all, val_preds, val_durations)
    print("Official ranking score:", json.dumps(rep["ranking_score"], default=float))
    for th, r in rep["by_theta"].items():
        print(f"  theta {th}: P {r['precision']:.3f} R {r['recall']:.3f} F1 {r['f1']:.3f}  FP/h {r['false_alarms_per_hour']:.1f}")

## Cell 7 - Test Set Evaluation & CSV Generation
`test_00 … test_04` → preprocessing → Stage 1 proposals → Stage 2 FPN decoder → Soft-NMS → `/content/drive/MyDrive/ABC27/submission_colab.csv`.

In [ ]:
all_dets = []
test_durations = {}
for sid in TEST_STREAM_IDS:                                        # test_00 ... test_04
    t0 = time.time()
    sd = preprocess_stream(sid)                                    # SVD clutter removal -> 12 bands -> IQR norm
    test_durations[sid] = sd.duration
    result = run_cascade(sd, stage1_model, stage2_model)           # Stage 1 proposals -> Stage 2 FPN decoder
    dets = detections_from_outputs(result, sid, sd.duration, score_floor=SOFT_NMS_FLOOR)   # decode + Soft-NMS
    all_dets += dets
    print(f"{sid}: {sd.duration / 60:6.1f} min | {len(result['regions']):3d} suspicious blocks "
          f"({sum(b - a for a, b in result['regions']) / sd.duration:4.0%} of stream) | "
          f"{len(dets):3d} falls | {time.time() - t0:.0f}s")
    del sd, result; gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

submission = pd.DataFrame(
    [{"stream_id": d["stream_id"], "t_start": round(d["start_time"], 3), "t_end": round(d["end_time"], 3),
      "score": round(d["score"], 6)} for d in all_dets],
    columns=["stream_id", "t_start", "t_end", "score"],
).sort_values(["stream_id", "t_start"]).reset_index(drop=True)
submission.to_csv(SUBMISSION_CSV, index=False)
print(f"\nSaved {len(submission)} detections -> {SUBMISSION_CSV}")

# ---- format check with the official validator (stream ids, t_start < t_end, inside the stream) ----
if abc_eval is not None:
    abc_eval.validate_predictions(SUBMISSION_CSV, test_durations)
    print("Official validate_predictions: OK")
submission.head(10)